# Sequencing Coverage Analysis

This notebook analyzes sequencing coverage from BAM/BED files:
- Coverage distribution
- Per-chromosome coverage
- Coverage uniformity
- Target region coverage (if applicable)

**Input**: Coverage file (BED, bedGraph, or mosdepth output)
**Outputs**: Coverage plots and statistics

In [ ]:
# Parameters
MIN_COVERAGE = 10  # Minimum coverage threshold
TARGET_COVERAGE = 30  # Target coverage for analysis
FIGURE_DPI = 150

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from galaxy_notebook import get, put, list_inputs

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = FIGURE_DPI

In [ ]:
# List available inputs
print("Available inputs:")
for inp in list_inputs():
    print(f"  [{inp['index']}] {inp['name']}")

In [ ]:
# Load coverage data
def load_coverage(filepath):
    """Load coverage data from various formats."""
    # Try to detect format
    with open(filepath, 'r') as f:
        first_line = f.readline().strip()
    
    # Check if it has header
    if first_line.startswith('#') or first_line.startswith('chrom'):
        df = pd.read_csv(filepath, sep='\t', comment='#')
    else:
        # Assume BED format: chrom, start, end, coverage
        df = pd.read_csv(filepath, sep='\t', header=None,
                        names=['chrom', 'start', 'end', 'coverage'])
    
    # Normalize column names
    df.columns = df.columns.str.lower()
    
    return df

input_path = get(0, format='path')
df = load_coverage(input_path)
print(f"Loaded {len(df):,} regions")
df.head()

In [ ]:
# Identify coverage column
cov_cols = ['coverage', 'depth', 'cov', 'count', 'value']
cov_col = None
for col in cov_cols:
    if col in df.columns:
        cov_col = col
        break

if cov_col is None:
    # Use last numeric column
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    cov_col = numeric_cols[-1]

print(f"Using coverage column: {cov_col}")

# Calculate region sizes if start/end available
if 'start' in df.columns and 'end' in df.columns:
    df['size'] = df['end'] - df['start']
else:
    df['size'] = 1

## 1. Coverage Distribution

In [ ]:
# Basic coverage statistics
coverage_values = df[cov_col]

stats = {
    'mean': coverage_values.mean(),
    'median': coverage_values.median(),
    'std': coverage_values.std(),
    'min': coverage_values.min(),
    'max': coverage_values.max(),
    'q25': coverage_values.quantile(0.25),
    'q75': coverage_values.quantile(0.75),
}

print("Coverage Statistics:")
for k, v in stats.items():
    print(f"  {k}: {v:.2f}")

In [ ]:
# Coverage distribution plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram
ax = axes[0, 0]
max_cov = min(coverage_values.quantile(0.99), 200)  # Cap for visualization
ax.hist(coverage_values.clip(upper=max_cov), bins=50, color='#3498db', edgecolor='white', alpha=0.7)
ax.axvline(stats['mean'], color='red', linestyle='--', label=f"Mean: {stats['mean']:.1f}")
ax.axvline(stats['median'], color='green', linestyle='--', label=f"Median: {stats['median']:.1f}")
ax.axvline(MIN_COVERAGE, color='orange', linestyle=':', label=f"Min threshold: {MIN_COVERAGE}")
ax.set_xlabel('Coverage')
ax.set_ylabel('Frequency')
ax.set_title('Coverage Distribution')
ax.legend()

# Cumulative distribution
ax = axes[0, 1]
sorted_cov = np.sort(coverage_values)
cumulative = np.arange(1, len(sorted_cov) + 1) / len(sorted_cov)
ax.plot(sorted_cov, cumulative, color='#3498db', linewidth=2)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.axvline(stats['median'], color='green', linestyle='--', alpha=0.7)
ax.set_xlabel('Coverage')
ax.set_ylabel('Cumulative Proportion')
ax.set_title('Cumulative Coverage Distribution')
ax.set_xlim(0, max_cov)

# Box plot
ax = axes[1, 0]
ax.boxplot(coverage_values, vert=True)
ax.axhline(MIN_COVERAGE, color='orange', linestyle=':', label=f"Min: {MIN_COVERAGE}")
ax.axhline(TARGET_COVERAGE, color='green', linestyle=':', label=f"Target: {TARGET_COVERAGE}")
ax.set_ylabel('Coverage')
ax.set_title('Coverage Box Plot')
ax.legend()

# Coverage thresholds
ax = axes[1, 1]
thresholds = [1, 5, 10, 15, 20, 30, 50, 100]
pct_above = [(coverage_values >= t).mean() * 100 for t in thresholds]
ax.bar(range(len(thresholds)), pct_above, color='#3498db', edgecolor='white')
ax.set_xticks(range(len(thresholds)))
ax.set_xticklabels([f'>={t}x' for t in thresholds])
ax.set_ylabel('Percentage of Regions')
ax.set_title('Coverage at Different Thresholds')
ax.set_ylim(0, 100)

# Add percentage labels
for i, p in enumerate(pct_above):
    ax.text(i, p + 2, f'{p:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
put(fig, 'coverage_distribution.png')
plt.show()

## 2. Per-Chromosome Coverage

In [ ]:
# Sort chromosomes naturally
def sort_chromosomes(chroms):
    def chrom_key(c):
        c = str(c).replace('chr', '')
        if c.isdigit():
            return (0, int(c))
        return (1, c)
    return sorted(chroms, key=chrom_key)

if 'chrom' in df.columns:
    # Per-chromosome statistics
    chrom_stats = df.groupby('chrom').agg({
        cov_col: ['mean', 'median', 'std', 'count'],
        'size': 'sum'
    }).round(2)
    chrom_stats.columns = ['mean_cov', 'median_cov', 'std_cov', 'n_regions', 'total_bp']
    
    # Sort chromosomes
    sorted_chroms = sort_chromosomes(chrom_stats.index.tolist())
    chrom_stats = chrom_stats.reindex(sorted_chroms)
    
    print("Per-chromosome coverage:")
    chrom_stats.head(10)

In [ ]:
# Plot per-chromosome coverage
if 'chrom' in df.columns:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # Mean coverage per chromosome
    ax = axes[0]
    x = range(len(chrom_stats))
    ax.bar(x, chrom_stats['mean_cov'], color='#3498db', edgecolor='white')
    ax.axhline(stats['mean'], color='red', linestyle='--', label=f"Overall mean: {stats['mean']:.1f}")
    ax.set_xticks(x)
    ax.set_xticklabels(chrom_stats.index, rotation=45, ha='right')
    ax.set_ylabel('Mean Coverage')
    ax.set_title('Mean Coverage per Chromosome')
    ax.legend()
    
    # Total bases per chromosome
    ax = axes[1]
    ax.bar(x, chrom_stats['total_bp'] / 1e6, color='#2ecc71', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(chrom_stats.index, rotation=45, ha='right')
    ax.set_ylabel('Total Bases (Mb)')
    ax.set_title('Sequenced Bases per Chromosome')
    
    plt.tight_layout()
    put(fig, 'chromosome_coverage.png')
    plt.show()

## 3. Coverage Uniformity

In [ ]:
# Calculate coverage uniformity metrics
# Coefficient of Variation (CV)
cv = stats['std'] / stats['mean'] if stats['mean'] > 0 else 0

# Interquartile Range normalized
iqr = stats['q75'] - stats['q25']
iqr_norm = iqr / stats['median'] if stats['median'] > 0 else 0

# Gini coefficient (measure of inequality)
def gini(values):
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    cumsum = np.cumsum(sorted_vals)
    return (n + 1 - 2 * np.sum(cumsum) / cumsum[-1]) / n if cumsum[-1] > 0 else 0

gini_coef = gini(coverage_values.values)

# Fold-80 penalty (how much coverage to capture 80% of targets)
pct80_cov = coverage_values.quantile(0.2)  # 80% of regions have at least this
fold80 = stats['mean'] / pct80_cov if pct80_cov > 0 else float('inf')

uniformity_metrics = {
    'CV (Coefficient of Variation)': cv,
    'Normalized IQR': iqr_norm,
    'Gini Coefficient': gini_coef,
    'Fold-80 Penalty': fold80
}

print("Coverage Uniformity Metrics:")
for k, v in uniformity_metrics.items():
    print(f"  {k}: {v:.3f}")

In [ ]:
# Lorenz curve for uniformity visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Lorenz curve
ax = axes[0]
sorted_cov = np.sort(coverage_values)
cumsum = np.cumsum(sorted_cov)
lorenz = cumsum / cumsum[-1] if cumsum[-1] > 0 else cumsum
x = np.linspace(0, 1, len(lorenz))

ax.plot(x, lorenz, 'b-', linewidth=2, label='Actual')
ax.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Perfect uniformity')
ax.fill_between(x, lorenz, x, alpha=0.3)
ax.set_xlabel('Cumulative Fraction of Regions')
ax.set_ylabel('Cumulative Fraction of Coverage')
ax.set_title(f'Lorenz Curve (Gini = {gini_coef:.3f})')
ax.legend()

# Uniformity bar chart
ax = axes[1]
metrics = list(uniformity_metrics.keys())
values = list(uniformity_metrics.values())
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
ax.barh(metrics, values, color=colors)
ax.set_xlabel('Value')
ax.set_title('Uniformity Metrics')

# Add value labels
for i, v in enumerate(values):
    ax.text(v + 0.02, i, f'{v:.3f}', va='center')

plt.tight_layout()
put(fig, 'coverage_uniformity.png')
plt.show()

## 4. Low Coverage Regions

In [ ]:
# Identify low coverage regions
low_cov = df[df[cov_col] < MIN_COVERAGE].copy()
n_low = len(low_cov)
pct_low = n_low / len(df) * 100

print(f"Regions with coverage < {MIN_COVERAGE}x: {n_low:,} ({pct_low:.2f}%)")

if n_low > 0 and 'size' in low_cov.columns:
    total_low_bp = low_cov['size'].sum()
    print(f"Total bases in low coverage regions: {total_low_bp:,} bp")

# Show top low coverage regions
if n_low > 0:
    print("\nLowest coverage regions:")
    low_cov.nsmallest(10, cov_col)

In [ ]:
# Low coverage by chromosome
if 'chrom' in df.columns and n_low > 0:
    low_by_chrom = low_cov.groupby('chrom').size()
    total_by_chrom = df.groupby('chrom').size()
    pct_low_by_chrom = (low_by_chrom / total_by_chrom * 100).fillna(0)
    
    # Sort
    sorted_chroms = sort_chromosomes(pct_low_by_chrom.index.tolist())
    pct_low_by_chrom = pct_low_by_chrom.reindex(sorted_chroms)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    pct_low_by_chrom.plot(kind='bar', ax=ax, color='#e74c3c', edgecolor='white')
    ax.set_ylabel(f'% Regions with < {MIN_COVERAGE}x Coverage')
    ax.set_xlabel('Chromosome')
    ax.set_title('Low Coverage Regions by Chromosome')
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    put(fig, 'low_coverage_by_chrom.png')
    plt.show()

## 5. Summary

In [ ]:
# Generate summary report
total_bases = df['size'].sum() if 'size' in df.columns else len(df)
bases_above_min = df[df[cov_col] >= MIN_COVERAGE]['size'].sum() if 'size' in df.columns else (df[cov_col] >= MIN_COVERAGE).sum()
bases_above_target = df[df[cov_col] >= TARGET_COVERAGE]['size'].sum() if 'size' in df.columns else (df[cov_col] >= TARGET_COVERAGE).sum()

summary = f"""
# Sequencing Coverage Analysis Summary

## Overview
- Total regions analyzed: {len(df):,}
- Total bases: {total_bases:,}

## Coverage Statistics
- Mean coverage: {stats['mean']:.2f}x
- Median coverage: {stats['median']:.2f}x
- Standard deviation: {stats['std']:.2f}x
- Range: {stats['min']:.0f}x - {stats['max']:.0f}x
- IQR: {stats['q25']:.1f}x - {stats['q75']:.1f}x

## Coverage Thresholds
- Bases >= {MIN_COVERAGE}x: {bases_above_min:,} ({bases_above_min/total_bases*100:.2f}%)
- Bases >= {TARGET_COVERAGE}x: {bases_above_target:,} ({bases_above_target/total_bases*100:.2f}%)

## Uniformity Metrics
- Coefficient of Variation: {cv:.3f}
- Gini Coefficient: {gini_coef:.3f}
- Fold-80 Penalty: {fold80:.2f}

## Quality Assessment
"""

# Quality assessment
if stats['mean'] >= TARGET_COVERAGE:
    summary += f"- Mean coverage ({stats['mean']:.1f}x) meets target ({TARGET_COVERAGE}x)\n"
else:
    summary += f"- WARNING: Mean coverage ({stats['mean']:.1f}x) below target ({TARGET_COVERAGE}x)\n"

if pct_low < 5:
    summary += f"- Good: Only {pct_low:.1f}% regions below {MIN_COVERAGE}x\n"
elif pct_low < 15:
    summary += f"- Moderate: {pct_low:.1f}% regions below {MIN_COVERAGE}x\n"
else:
    summary += f"- WARNING: {pct_low:.1f}% regions below {MIN_COVERAGE}x\n"

if cv < 0.5:
    summary += f"- Good uniformity (CV = {cv:.3f})\n"
else:
    summary += f"- Variable coverage (CV = {cv:.3f})\n"

print(summary)
put(summary, 'coverage_summary.md')

In [ ]:
# Save statistics tables
stats_df = pd.DataFrame({
    'Metric': ['Mean', 'Median', 'Std Dev', 'Min', 'Max', 'Q25', 'Q75'],
    'Value': [stats['mean'], stats['median'], stats['std'], 
              stats['min'], stats['max'], stats['q25'], stats['q75']]
})
put(stats_df, 'coverage_statistics.tsv')

if 'chrom' in df.columns:
    put(chrom_stats.reset_index(), 'chromosome_coverage.tsv')

if n_low > 0:
    put(low_cov.head(1000), 'low_coverage_regions.tsv')

print("\n=== Analysis Complete ===")
print("Generated outputs:")
print("  - coverage_distribution.png")
print("  - chromosome_coverage.png")
print("  - coverage_uniformity.png")
print("  - low_coverage_by_chrom.png")
print("  - coverage_summary.md")
print("  - coverage_statistics.tsv")
print("  - chromosome_coverage.tsv")
print("  - low_coverage_regions.tsv")